# SatQuery AI — Division 2: Single-Image Remote-Sensing Intelligence
## Google Colab GPU Compute Pipeline & Scientific Training Runner

- **Division**: Division 2 (Single-Image Remote-Sensing Intelligence: VQA + Visual Grounding)
- **Lead Owner**: Sruthi (`sruthi-270` / `rajamanurisruthi@gmail.com`)
- **Branch**: `feature/sruthi-single-image`
- **Target Base Model**: `google/paligemma-3b-pt-224`
- **Status**: `[PHASE 2 SMOKE TEST VERIFIED — RUNNING FULL TRAINING]`

> **Notice**: This notebook runs exclusively as an external GPU compute worker. The final SatQuery application runtime does not depend on Colab.

### Step 1: GPU Compute Environment & Hardware Diagnostics

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:    {torch.version.cuda}")

### Step 2: Secure Hugging Face Authentication

In [ ]:
import os
import huggingface_hub

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    hf_token = getpass.getpass('Enter Hugging Face Access Token (Read role): ')

huggingface_hub.login(token=hf_token)
print("Hugging Face authentication completed securely.")

### Step 3: Fast Git Repository Checkout & Dependency Setup

In [ ]:
import os, sys
if not os.path.exists('/content/SatQuery'):
    !git clone https://github.com/Lalith2007/SatQuery.git /content/SatQuery
%cd /content/SatQuery
!git fetch origin
!git checkout feature/sruthi-single-image
!git reset --hard origin/feature/sruthi-single-image

# Fix Colab torchao conflict and install dependencies
!pip uninstall -y torchao
!pip install tqdm fastapi pydantic-settings tifffile pytest-asyncio peft
!pip install -e . --no-deps

if '/content/SatQuery' not in sys.path:
    sys.path.insert(0, '/content/SatQuery')

print("✓ Environment and repository ready!")

### Step 4: Phase 1 — Real Model Load & Generation Verification

In [ ]:
import torch, gc
from PIL import Image
from transformers import PaliGemmaForConditionalGeneration

model_id = "google/paligemma-3b-pt-224"

print(f"Loading {model_id} on GPU...")
try:
    from transformers import PaliGemmaProcessor
    processor = PaliGemmaProcessor.from_pretrained(model_id)
except Exception:
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained(model_id)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    device_map="cuda:0" if torch.cuda.is_available() else None,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"SUCCESS: Loaded {model_id} on {model.device}!")
print(f"Total Model Parameters: {total_params:,}")

# Test genuine model.generate() output
test_img = Image.new("RGB", (224, 224), color=(34, 139, 34))
inputs = processor(text="<image>answer en What is the dominant land cover?", images=test_img, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=32)
ans = processor.decode(output[0], skip_special_tokens=True)
print(f"Genuine Model Generation Output: '{ans}'")
print("PHASE 1 PASSED: REAL_MODEL_LOADED = True")

# Free Step 4 model memory so Step 5/6 has full GPU memory available
del model, processor
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed for Phase 2 & 3 training.")

### Step 5: Phase 2 — Real LoRA Gradient & Backprop Smoke Test (Verified Proof)

In [ ]:
# Runs genuine forward pass, loss.backward(), non-zero gradient check, and optimizer.step() parameter delta
!python3 specialists/single_image/adaptation/train_lora.py --smoke-test --device cuda

### Step 6: Phase 3 — Real LoRA Domain Adaptation Training (3 Epochs with live loss & progress bar)

In [ ]:
# Execute real gradient-based LoRA training with live progress bar and step logging
!python3 specialists/single_image/adaptation/train_lora.py --epochs 3 --train-count 150 --val-count 30 --device cuda

### Step 7: Phase 4 & 5 — Real Model Evaluation & Synchronized CUDA Latency Benchmark

In [ ]:
# Strict scientific evaluation on N=150 held-out samples (Zero fallback permitted)
!python3 specialists/single_image/evaluation/reproducibility.py

### Step 8: Phase 6 — Export Reproducibility Manifest & Artifact Archive

In [ ]:
!python3 specialists/single_image/colab/reproducibility_manifest.py
!tar -czvf satquery_division2_adapter_package.tar.gz specialists/single_image/weights/ specialists/single_image/evaluation/ specialists/single_image/colab/
print("Artifact bundle generated: satquery_division2_adapter_package.tar.gz")